# 01 — Tabularizing Time Series: The Foundation of ML-Based Forecasting

**Companion notebook to Article 1 of the *Time Series Feature Engineering* series.**

📖 Read the article: [Medium](https://medium.com/@asidd24) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

This notebook demonstrates:

1. Why time series data cannot be fed directly to ML models
2. Tabularization — converting a 1D sequence into a 2D feature matrix
3. The three categories of features: **lag**, **window**, and **datetime**
4. Temporal leakage — the #1 silent killer, with a live before/after demo
5. A complete single-step forecasting pipeline with LightGBM
6. Bridging forecasting to anomaly detection using the same features


## 1. Setup

In [ ]:
import sys
sys.path.append('..')  # Allow importing from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reusable utilities from this repo
from src.data import generate_daily_sales, time_train_test_split
from src.features import (
    add_lag_features,
    add_rolling_features,
    add_datetime_features,
    add_cyclical_encoding,
    build_forecasting_feature_matrix,
)
from src.metrics import mae, rmse, wmape, vandeput_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
np.random.seed(42)

## 2. The Sequential vs Tabular Problem

Time series data arrives as a **sequence**: an ordered stream of observations where each value depends on the ones before it.

ML models like XGBoost, LightGBM, and Random Forests expect **independent observations** — rows that don't have a temporal relationship.

These two structures are fundamentally incompatible. Let's see this.

In [ ]:
# Generate 2 years of daily sales
sales = generate_daily_sales(n_days=730, seed=42)
print(f"Series shape:  {sales.shape}")
print(f"Date range:    {sales.index.min().date()} to {sales.index.max().date()}")
print(f"\nFirst 10 observations:")
sales.head(10)

In [ ]:
# Visualize the raw series
fig, ax = plt.subplots(figsize=(14, 4))
sales.plot(ax=ax, linewidth=1)
ax.set_title('Raw Time Series — 2 Years of Daily Sales')
ax.set_ylabel('Sales')
plt.tight_layout()
plt.show()

**This is a 1D sequence.** We cannot fit XGBoost on this directly — it needs a 2D matrix where each row is an independent training instance.

## 3. Lag Features (Endogenous)

Lag features are the past values of the target, used as inputs to predict the future.

- `lag_1` → yesterday's sales
- `lag_7` → sales one week ago (weekly seasonality)
- `lag_14` → sales two weeks ago
- `lag_28` → sales four weeks ago (monthly pattern)

In [ ]:
df = sales.to_frame(name='target')
df = add_lag_features(df, 'target', lags=[1, 2, 3, 7, 14, 28])

df.head(30)

In [ ]:
# Show what happened
print("Notice the NaN pattern:")
print(f"  target_lag_1  needs 1 previous obs  → 1 NaN row")
print(f"  target_lag_28 needs 28 previous obs → 28 NaN rows")
print(f"\nAfter dropping NaN rows, we have {df.dropna().shape[0]} usable training instances.")

## 4. Window Features (Rolling Statistics)

Rather than raw lag values, we can encode **statistical summaries** of recent history:

- Rolling mean over 7 days → short-term average
- Rolling std over 14 days → recent volatility  
- Rolling max over 28 days → recent peak

**Critical:** Every rolling operation must be preceded by `.shift(min_horizon)` to prevent target leakage. Our `add_rolling_features` enforces this automatically.

In [ ]:
df = add_rolling_features(
    df,
    target_col='target',
    windows=[7, 14, 28],
    functions=['mean', 'std'],
    min_horizon=1,   # No leakage — window ends at t-1
)

df.head(30)

## 5. Datetime Features

Extracted directly from the timestamp — always future-known, so they never cause leakage.

In [ ]:
df = add_datetime_features(
    df,
    features=['dayofweek', 'month', 'quarter', 'is_weekend'],
)

# Encode dayofweek cyclically so the model learns Sunday and Monday are adjacent
df = add_cyclical_encoding(df, 'dayofweek', period=7, drop_original=False)
df = add_cyclical_encoding(df, 'month', period=12, drop_original=False)

df.head()

## 6. Temporal Leakage — The Live Demo

Let's demonstrate the #1 silent killer in time series ML with a side-by-side comparison.

**Scenario:** Same data, same model, same everything — except one pipeline has a leakage bug in its rolling feature.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit

# --- LEAKY pipeline: rolling mean WITHOUT shift ---
df_leaky = sales.to_frame(name='target').copy()
df_leaky['lag_1'] = sales.shift(1)
df_leaky['lag_7'] = sales.shift(7)
df_leaky['rolling_mean_7'] = sales.rolling(7).mean()  # ❌ leaks target
df_leaky = df_leaky.dropna()

# --- CLEAN pipeline: rolling mean WITH shift(1) ---
df_clean = sales.to_frame(name='target').copy()
df_clean['lag_1'] = sales.shift(1)
df_clean['lag_7'] = sales.shift(7)
df_clean['rolling_mean_7'] = sales.shift(1).rolling(7).mean()  # ✅ safe
df_clean = df_clean.dropna()

# Evaluate both with time-aware CV
def cv_score(df, target_col='target'):
    X = df.drop(columns=target_col)
    y = df[target_col]
    tscv = TimeSeriesSplit(n_splits=5)
    maes = []
    for train_idx, test_idx in tscv.split(X):
        model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[test_idx])
        maes.append(mae(y.iloc[test_idx].values, pred))
    return np.mean(maes), np.std(maes)

mae_leaky, std_leaky = cv_score(df_leaky)
mae_clean, std_clean = cv_score(df_clean)

print("=" * 55)
print("TEMPORAL LEAKAGE DEMONSTRATION")
print("=" * 55)
print(f"❌ Leaky pipeline  CV MAE: {mae_leaky:.3f} ± {std_leaky:.3f}")
print(f"✅ Clean pipeline  CV MAE: {mae_clean:.3f} ± {std_clean:.3f}")
print()
print(f"Leakage makes the model look {(mae_clean/mae_leaky - 1)*100:.1f}% BETTER than it really is.")
print("In production, this 'better' model degrades sharply because it")
print("never actually learned to predict the future — it learned to peek.")

## 7. The Complete Pipeline

Everything above, in one leakage-safe call using the repo's `build_forecasting_feature_matrix` utility.

In [ ]:
# One line — full leakage-safe feature matrix
df_features = build_forecasting_feature_matrix(
    sales,
    lags=[1, 2, 3, 7, 14, 28],
    rolling_windows=[7, 14, 28],
    rolling_functions=['mean', 'std'],
    forecast_horizon=1,
    include_datetime=True,
)

print(f"Feature matrix shape: {df_features.shape}")
print(f"Features: {[c for c in df_features.columns if c != 'target']}")

In [ ]:
# Time-aware train/test split
train_df, test_df = time_train_test_split(df_features, test_size=0.2)

X_train = train_df.drop(columns='target')
y_train = train_df['target']
X_test = test_df.drop(columns='target')
y_test = test_df['target']

print(f"Train: {len(X_train)} rows | {X_train.index.min().date()} → {X_train.index.max().date()}")
print(f"Test:  {len(X_test)} rows | {X_test.index.min().date()} → {X_test.index.max().date()}")

In [ ]:
# Fit a gradient boosting model
model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluate with multiple metrics
print("Test Metrics")
print("-" * 30)
print(f"MAE:            {mae(y_test.values, y_pred):.2f}")
print(f"RMSE:           {rmse(y_test.values, y_pred):.2f}")
print(f"wMAPE:          {wmape(y_test.values, y_pred):.3%}")
print(f"Vandeput Score: {vandeput_score(y_test.values, y_pred):.2f}")

In [ ]:
# Visualize predictions
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_test.index, y_test.values, label='Actual', linewidth=2, color='steelblue')
ax.plot(y_test.index, y_pred, label='Predicted', linestyle='--', linewidth=2, color='darkorange')
ax.set_title('Single-Step Forecast on Held-Out Test Period')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — which features drove the predictions?
importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importances.tail(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Feature Importances (GradientBoostingRegressor)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Bridging to Anomaly Detection

The same tabularization framework powers anomaly detection. The pattern:

1. Fit a forecasting model on the tabularized series
2. Compute residuals: `actual - predicted`
3. Set a threshold on the residual distribution (typically 3σ from the mean)
4. Flag observations with residuals exceeding the threshold

In [ ]:
# Compute residuals on the full series
y_all = df_features['target']
X_all = df_features.drop(columns='target')
residuals = y_all - model.predict(X_all)

# Establish normal residual distribution using training residuals
train_residuals = residuals.loc[y_train.index]
threshold_upper = train_residuals.mean() + 3 * train_residuals.std()
threshold_lower = train_residuals.mean() - 3 * train_residuals.std()

# Flag anomalies
anomaly_mask = (residuals > threshold_upper) | (residuals < threshold_lower)
anomalies = residuals[anomaly_mask]

print(f"Detected {len(anomalies)} anomalies out of {len(residuals)} observations")
print(f"Anomaly rate: {len(anomalies) / len(residuals):.2%}")
print(f"Threshold: [{threshold_lower:.2f}, {threshold_upper:.2f}]")

In [ ]:
# Visualize anomalies on the original series
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: original series with anomalies highlighted
axes[0].plot(y_all.index, y_all.values, linewidth=1, color='steelblue', label='Sales')
axes[0].scatter(anomalies.index, y_all.loc[anomalies.index],
                color='crimson', s=40, zorder=5, label=f'Anomalies (n={len(anomalies)})')
axes[0].set_title('Anomaly Detection via Forecast Residuals')
axes[0].set_ylabel('Sales')
axes[0].legend()

# Bottom: residuals with threshold bands
axes[1].plot(residuals.index, residuals.values, linewidth=0.8, color='gray')
axes[1].axhline(threshold_upper, color='crimson', linestyle='--', linewidth=1, label='±3σ threshold')
axes[1].axhline(threshold_lower, color='crimson', linestyle='--', linewidth=1)
axes[1].scatter(anomalies.index, anomalies.values, color='crimson', s=30, zorder=5)
axes[1].set_ylabel('Residual')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Key Takeaways

- Time series must be **tabularized** before standard ML algorithms can consume it
- The feature matrix has three sources: **lag features**, **window features**, and **datetime features**
- **Temporal leakage** is the #1 silent killer — always `.shift()` before rolling, always split by time
- The same feature matrix drives both **forecasting** and **anomaly detection**

---

## Next Notebook

**`02_ml_pipeline_leakage.ipynb`** covers:
- Building a leakage-safe sklearn pipeline with `feature-engine`
- Walk-forward cross-validation with `TimeSeriesSplit`
- The three multi-step forecasting strategies: recursive, direct, multi-output

**→ Article 2: *Why Time Series Breaks Your ML Pipeline (And How to Fix It)***
